# **--- GETAROUND PROJECT (pricing optimization) PREPROCESSING ---** #

## **1. Libraries import** ##

In [21]:
import pandas as pd
import numpy as np
import os
import json

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats.mstats import winsorize

import joblib

## **2. Data import and preparation for preprocessing** ##

_Import the clean dataset_

In [22]:
df_pricing=pd.read_csv("../DATA/Data_intermediate/df_pricing_clean.csv")

_Separating target from features_

In [23]:
target='rental_price_per_day'
y=df_pricing[target].copy()
X=df_pricing.drop(columns=target).copy()

_Separating features per type_

In [24]:
X_num=X.select_dtypes(include=["int64"]).copy()
X_cat=X.select_dtypes(include="object").copy()
X_bool=X.select_dtypes(include="bool").copy()

# Lists of column names : 
X_num_names=X_num.columns.to_list()
X_cat_names=X_cat.columns.to_list()
X_bool_names=X_bool.columns.to_list()

## **3. Transformers per type of features** ##

In [25]:
numeric_transformer = Pipeline([
    ('scaler', StandardScaler()),
    ])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

boolean_transformer = Pipeline([
    ('bool_to_int', OrdinalEncoder())  # Transform the booleans True/False into 1/0
])

## **4. Train test split** ##

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **5. Applying the preprocessing pipeline** ##

Creation of the preprocessing pipeline

In [27]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, X_num_names),
    ('cat', categorical_transformer, X_cat_names),
    ('bool', boolean_transformer, X_bool_names)
])

Application of the preprocessing pipeline

In [28]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

## **6. Saving the pipeline and the transformed datasets** ##

In [29]:
# Saving the preprocessing pipeline
joblib.dump(preprocessor,'../MODELS/preprocessing_pipeline.pkl')


# Saving the transformed dataset after applying the pipeline into a light and fast binary file .npy
np.save('../DATA/Data_processed/X_train_preprocessed.npy', X_train_preprocessed)
np.save('../DATA/Data_processed/X_test_preprocessed.npy', X_test_preprocessed)
np.save('../DATA/Data_processed/y_train.npy', y_train.to_numpy()) # pandas series treansormed into a Numpy array
np.save('../DATA/Data_processed/y_test.npy', y_test.to_numpy())
